
# Task B — SOC Automation Demo

This notebook loads the best artifact bundle exported by Task A and provides an interactive SOC-style triage helper with IOC extraction, classification, and lightweight explainability.


In [ ]:

# Cell 1 — Install dependencies (quiet)
!pip install -q pandas numpy scikit-learn torch transformers beautifulsoup4 lxml joblib ipywidgets


In [ ]:

# Cell 2 — Imports, configuration, and artifact discovery
import json
import os
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from html import unescape
from joblib import load

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ARTIFACT_ROOT = Path('/kaggle/working/artifacts/best_model')
INPUT_ARTIFACT_ROOT = Path('/kaggle/input')
DEFAULT_TRANSFORMER_DIR = Path('/kaggle/working/artifacts/distilbert')

if not ARTIFACT_ROOT.exists():
    log_path = None
    for dataset_dir in INPUT_ARTIFACT_ROOT.glob('*'):
        candidate = dataset_dir / 'best_model'
        if candidate.exists():
            ARTIFACT_ROOT = candidate
            log_path = dataset_dir
            break
    if ARTIFACT_ROOT.exists():
        print(f"[LOG] Loaded artifact bundle from {ARTIFACT_ROOT} (dataset: {log_path})")
    else:
        raise FileNotFoundError('Could not find best_model artifacts. Run Task A first or attach the exported bundle as a Kaggle dataset.')
else:
    print(f"[LOG] Using artifact bundle from {ARTIFACT_ROOT}")

with open(ARTIFACT_ROOT / 'preproc_config.json', 'r') as f:
    PREPROC_CONFIG = json.load(f)

print('[LOG] Preprocessing config loaded:')
print(json.dumps(PREPROC_CONFIG, indent=2))

VECTORIZER_PATH = ARTIFACT_ROOT / PREPROC_CONFIG.get('vectorizer', 'vectorizer.joblib')
MODEL_PATH = ARTIFACT_ROOT / PREPROC_CONFIG.get('model', 'model.joblib')
TRANSFORMER_RELATIVE = PREPROC_CONFIG.get('transformer_path')

vectorizer = load(VECTORIZER_PATH)
model = load(MODEL_PATH)

transformer_available = False
transformer_model = None
transformer_tokenizer = None

if TRANSFORMER_RELATIVE:
    transformer_dir = (ARTIFACT_ROOT / TRANSFORMER_RELATIVE).resolve()
    if transformer_dir.exists():
        transformer_tokenizer = AutoTokenizer.from_pretrained(transformer_dir)
        transformer_model = AutoModelForSequenceClassification.from_pretrained(transformer_dir)
        transformer_available = True
    elif DEFAULT_TRANSFORMER_DIR.exists():
        transformer_tokenizer = AutoTokenizer.from_pretrained(DEFAULT_TRANSFORMER_DIR)
        transformer_model = AutoModelForSequenceClassification.from_pretrained(DEFAULT_TRANSFORMER_DIR)
        transformer_available = True

CLASSIFIER_THRESHOLD = 0.5


In [ ]:

# Cell 3 — IOC extraction utilities
IOC_PATTERNS = {
    'urls': re.compile(r'(https?://[\w\-./?%&=+#]+)', re.IGNORECASE),
    'domains': re.compile(r'([a-z0-9-]+\.[a-z]{2,})', re.IGNORECASE),
    'ipv4': re.compile(r'(?:[0-9]{1,3}\.){3}[0-9]{1,3}'),
    'emails': re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'),
    'hashes': re.compile(r'[a-f0-9]{32,64}', re.IGNORECASE),
}


def extract_iocs(text: str) -> dict:
    findings = {}
    for name, pattern in IOC_PATTERNS.items():
        matches = sorted(set(pattern.findall(text)))
        if matches:
            findings[name] = matches
    return findings


In [ ]:

# Cell 4 — Preprocessing and scoring helpers
from typing import Dict, Any

URL_TOKEN = '<URL>'
NUM_TOKEN = '<NUM>'


def strip_html(text: str) -> str:
    soup = BeautifulSoup(text, 'lxml')
    for tag in soup(['script', 'style']):
        tag.decompose()
    return soup.get_text(separator=' ').strip()


def normalize_text(text: str, lower: bool = True, url_token: str = URL_TOKEN, num_token: str = NUM_TOKEN) -> str:
    text = unescape(text)
    if lower:
        text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', url_token, text)
    text = re.sub(r'[0-9]+', num_token, text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def preprocess_text(subject: str = '', body: str = '', is_html: bool = False) -> str:
    subject = subject or ''
    body = body or ''
    combined = f"{subject}
{body}".strip()
    if is_html:
        combined = strip_html(combined)
    return normalize_text(combined)


def score_text(subject: str = '', body: str = '', is_html: bool = False) -> Dict[str, Any]:
    clean_text = preprocess_text(subject, body, is_html)
    score = None

    if transformer_available and transformer_model is not None:
        inputs = transformer_tokenizer(clean_text, return_tensors='pt', truncation=True, max_length=256, padding='max_length')
        with torch.no_grad():
            outputs = transformer_model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)[0]
            score = float(probabilities[1])
    else:
        features = vectorizer.transform([clean_text])
        if hasattr(model, 'predict_proba'):
            score = float(model.predict_proba(features)[0, 1])
        else:
            decision = model.decision_function(features)
            score = float(1 / (1 + math.exp(-decision[0])))

    label = int(score >= CLASSIFIER_THRESHOLD)
    return {'label': label, 'score': score, 'clean_text': clean_text}


def explain_text(clean_text: str, top_k: int = 10) -> Dict[str, Any]:
    explanation = {'supports': [], 'refutes': []}
    if hasattr(model, 'coef_') and hasattr(vectorizer, 'vocabulary_'):
        feature_names = np.array(vectorizer.get_feature_names_out())
        coef = model.coef_[0]
        word_to_coef = {feature_names[i]: coef[i] for i in range(len(feature_names))}
        tokens = clean_text.split()
        token_scores = [(token, word_to_coef.get(token, 0.0)) for token in tokens if token in word_to_coef]
        token_scores.sort(key=lambda x: abs(x[1]), reverse=True)
        for token, weight in token_scores[:top_k]:
            bucket = 'supports' if weight >= 0 else 'refutes'
            explanation[bucket].append({'token': token, 'weight': float(weight)})
    elif transformer_available:
        explanation['info'] = 'Transformer-based model loaded; detailed token attributions are not available in this lightweight demo.'
    else:
        explanation['info'] = 'Explainability unavailable for this model type.'
    return explanation


In [ ]:

# Cell 5 — Interactive SOC widget
import ipywidgets as widgets
from IPython.display import display, JSON

subject_widget = widgets.Text(description='Subject', placeholder='Optional subject line')
body_widget = widgets.Textarea(description='Body', layout=widgets.Layout(width='100%', height='200px'))
html_checkbox = widgets.Checkbox(value=False, description='Body is HTML?')
classify_button = widgets.Button(description='Classify', button_style='primary')
output_area = widgets.Output()

def on_classify(_):
    output_area.clear_output()
    with output_area:
        result = score_text(subject_widget.value, body_widget.value, html_checkbox.value)
        iocs = extract_iocs(result['clean_text'])
        explanation = explain_text(result['clean_text'])
        payload = {
            'prediction': {'label': int(result['label']), 'probability': float(result['score'])},
            'iocs': iocs,
            'explanation': explanation,
        }
        display(JSON(payload))

classify_button.on_click(on_classify)

ui = widgets.VBox([
    widgets.HTML('<h3>Phishing Email Analyzer</h3>'),
    subject_widget,
    body_widget,
    html_checkbox,
    classify_button,
    output_area,
])

display(ui)


In [ ]:

# Cell 6 — Batch scoring helper
import io

upload_widget = widgets.FileUpload(accept='.csv', multiple=False)
batch_output = widgets.Output()

def on_upload_change(change):
    batch_output.clear_output()
    if not upload_widget.value:
        return
    (_, file_info), = upload_widget.value.items()
    buffer = io.BytesIO(file_info['content'])
    df = pd.read_csv(buffer).fillna('')
    with batch_output:
        print(f"[LOG] Loaded CSV with shape {df.shape}")
        results = []
        for _, row in df.iterrows():
            result = score_text(row.get('subject', ''), row.get('body', ''), row.get('body_is_html', False))
            iocs = extract_iocs(result['clean_text'])
            results.append({
                'subject': row.get('subject', ''),
                'label': int(result['label']),
                'probability': float(result['score']),
                'iocs': iocs,
            })
        out_df = pd.DataFrame(results)
        output_path = Path('/kaggle/working/soc_scored.csv')
        out_df.to_csv(output_path, index=False)
        print(f"[LOG] Saved batch scoring results to {output_path}")
        display(out_df.head())

upload_widget.observe(on_upload_change, names='value')

display(widgets.VBox([
    widgets.HTML('<h4>Batch scoring (upload CSV with subject/body/body_is_html columns)</h4>'),
    upload_widget,
    batch_output,
]))


In [ ]:

# Cell 7 — Example runs
examples = [
    {
        'subject': 'Payroll Update',
        'body': 'Hi team, please review the attached payroll spreadsheet before Friday. Thanks!',
        'body_is_html': False,
    },
    {
        'subject': 'URGENT: Reset your account password now',
        'body': 'Your account has been compromised. Visit http://secure-login-example.com immediately to verify your credentials or your account will be terminated.',
        'body_is_html': False,
    },
]

for example in examples:
    result = score_text(example['subject'], example['body'], example['body_is_html'])
    iocs = extract_iocs(result['clean_text'])
    explanation = explain_text(result['clean_text'])
    print('=' * 60)
    print(f"Subject: {example['subject']}")
    print(f"Label: {result['label']} | Probability: {result['score']:.3f}")
    print(f"IOCs: {iocs}")
    if 'info' in explanation:
        print(f"Explainability: {explanation['info']}")
    else:
        supports = explanation.get('supports', [])
        refutes = explanation.get('refutes', [])
        print(f"Top supporting tokens: {supports[:5]}")
        print(f"Top refuting tokens: {refutes[:5]}")
